# Robotarmen: koordinater, lokal kinematikk og koblet dynamikk

## Pilotprosjekt for Matematikk 1

En robotarm må kunne tolke posisjoner i ulike koordinatsystemer, omsette små bevegelser av gripepunktet til bevegelser i leddene og bevege seg stabilt mot en ønsket arbeidsstilling.

I dette prosjektet bruker vi en plan robotarm med to roterende ledd. Robotkinematikken er anvendelseskonteksten, mens hovedtemaene er

- lineære transformasjoner,
- løsning av lineære ligningssystemer,
- determinant og inverterbarhet,
- egenverdier og egenvektorer,
- variabelbytte,
- koblede differensialligninger,
- Eulers metode for vektorsystemer.

Prosjektet har fire deler:

1. **Koordinatkalibrering:** fra kamerakoordinater til robotkoordinater
2. **Lokal kinematikk:** små leddendringer og små gripepunktbevegelser
3. **Leddynamikk:** et koblet andreordens ODE-system
4. **Normale moder og lokal posisjonsstyring:** diagonalisering og variabelbytte

### Viktig avgrensning

Studentene skal ikke utlede en Jacobimatrise med flervariabel kjerneregel. Hastighets- og forskyvningsmatrisen $J$ gis som en modellformel. Vi bruker den som en vanlig $2\times2$-matrise.

### Mulig nedkorting senere

Et kortere prosjekt kan begrenses til

- A1–A4,
- B1–B4,
- C1–C4.

Del D og enkelte parameterstudier kan flyttes til fordypning.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Robotmodellen

Robotarmen har leddlengder $L_1$ og $L_2$ og leddvinkler $\theta_1,\theta_2$.

```text
                         gripepunkt P
                              o
                             /
                         L2 /
                           /
                    albue o
                         /
                     L1 /
                       /
                base o
```

Endepunktet er

$$
\boxed{
\begin{aligned}
x&=L_1\cos\theta_1+L_2\cos(\theta_1+\theta_2),\\
y&=L_1\sin\theta_1+L_2\sin(\theta_1+\theta_2).
\end{aligned}}
$$

Dette kalles framoverkinematikk: kjente leddvinkler gir kjent gripepunkt.

In [ ]:
L1 = 0.80
L2 = 0.60


def framoverkinematikk(theta):
    theta1, theta2 = theta
    albue = np.array([
        L1*np.cos(theta1),
        L1*np.sin(theta1)
    ])
    gripepunkt = albue + np.array([
        L2*np.cos(theta1 + theta2),
        L2*np.sin(theta1 + theta2)
    ])
    return albue, gripepunkt


def tegn_robot(theta, ax=None, label=None):
    if ax is None:
        fig, ax = plt.subplots()
    albue, P = framoverkinematikk(theta)
    ax.plot([0, albue[0], P[0]], [0, albue[1], P[1]], "o-", label=label)
    ax.axis("equal")
    ax.grid()
    return ax

# Del A: Fra kamerakoordinater til robotkoordinater

Et kamera ser arbeidsbordet i bildekoordinater $(u,v)$. Robotarmen bruker bordkoordinater $(x,y)$.

Vi bruker en affin modell:

$$
\boxed{
\begin{pmatrix}x\\y\end{pmatrix}
=
\begin{pmatrix}a&b\\c&d\end{pmatrix}
\begin{pmatrix}u\\v\end{pmatrix}
+
\begin{pmatrix}e\\f\end{pmatrix}.}
$$

I homogene koordinater blir dette én matrisemultiplikasjon:

$$
\boxed{
\begin{pmatrix}x\\y\\1\end{pmatrix}
=
H
\begin{pmatrix}u\\v\\1\end{pmatrix},
\qquad
H=
\begin{pmatrix}
a&b&e\\c&d&f\\0&0&1
\end{pmatrix}.}
$$

## A.1 Kalibrering med tre markører

Tre markører er målt både i kamera- og robotkoordinater.

For $x$-koordinaten får vi

$$
\begin{pmatrix}
u_1&v_1&1\\u_2&v_2&1\\u_3&v_3&1
\end{pmatrix}
\begin{pmatrix}a\\b\\e\end{pmatrix}
=
\begin{pmatrix}x_1\\x_2\\x_3\end{pmatrix}.
$$

Et tilsvarende system bestemmer $c,d,f$ fra $y$-koordinatene.

## Oppgave A1: Finn kalibreringsmatrisen

Løs de to $3\times3$-systemene og sett sammen $H$.

In [ ]:
kamera = np.array([
    [120.0,  80.0],
    [520.0, 100.0],
    [160.0, 420.0]
])

robot = np.array([
    [0.20, 0.15],
    [1.05, 0.20],
    [0.25, 0.85]
])

A_kal = np.column_stack([kamera, np.ones(3)])

koeff_x = ...
koeff_y = ...

H = np.array([
    [koeff_x[0], koeff_x[1], koeff_x[2]],
    [koeff_y[0], koeff_y[1], koeff_y[2]],
    [0.0,        0.0,        1.0]
])

print("Kalibreringsmatrise H:
", H)
print("det(A_kal) =", ...)

## Oppgave A2: Kontroller kalibreringspunktene

Transformer alle tre kameramarkørene. De skal gi robotkoordinatene tilbake, bortsett fra avrundingsfeil.

In [ ]:
kamera_h = np.column_stack([kamera, np.ones(3)])
robot_beregnet_h = ...
robot_beregnet = ...

print("Beregnet:
", robot_beregnet)
print("Fasit:
", robot)
print("Maksimal feil:", ...)

## A.2 Transformer et nytt objekt

Kameraet har funnet sentrum av en komponent i punktet

$$p_K=(340,260).$$

Bruk $H$ til å finne punktet i robotkoordinater.

In [ ]:
p_kamera = np.array([340.0, 260.0, 1.0])
p_robot_h = ...
p_robot = ...

print("Objektets robotkoordinater:", p_robot)

## Oppgave A3: Den inverse transformasjonen

Dersom $H$ er inverterbar, kan et robotpunkt omformes tilbake til kamerakoordinater:

$$p_K=H^{-1}p_R.$$

Beregn $H^{-1}$ og kontroller det nye punktet.

In [ ]:
H_inv = ...
p_kamera_tilbake = ...

print("Tilbakeført kamerapunkt:", p_kamera_tilbake)
print("Kontroll H_inv H:
", ...)

## Oppgave A4: Dårlig kalibreringsgeometri

Lag tre kalibreringspunkter som nesten ligger på samme rette linje.

1. Beregn determinanten til kalibreringsmatrisen.
2. Legg en liten målefeil til ett punkt.
3. Beregn kalibreringen på nytt.
4. Sammenlign endringen med den opprinnelige kalibreringen.

Forklar hvorfor kalibreringspunktene bør dekke en todimensjonal del av arbeidsområdet.

In [ ]:
kamera_dårlig = np.array([
    [100.0, 100.0],
    [300.0, 201.0],
    [500.0, 300.0]
])

A_dårlig = np.column_stack([kamera_dårlig, np.ones(3)])
print("det(A_dårlig) =", ...)

# Gjennomfør en liten feilfølsomhetsstudie.

# Del B: Lokal kinematikk som lineær algebra

Vi velger en kjent arbeidsstilling

$$
\theta^*=
\begin{pmatrix}\theta_1^*\\\theta_2^*\end{pmatrix}.
$$

For små endringer bruker vi den lokale lineære modellen

$$
\boxed{
\Delta p\approx J^*\Delta\theta,}
$$

hvor

$$
\Delta p=
\begin{pmatrix}\Delta x\\\Delta y\end{pmatrix},
\qquad
\Delta\theta=
\begin{pmatrix}\Delta\theta_1\\\Delta\theta_2\end{pmatrix}.
$$

Matrisen $J^*$ gis av

$$
\boxed{
J(\theta_1,\theta_2)=
\begin{pmatrix}
-L_1\sin\theta_1-L_2\sin(\theta_1+\theta_2)&-L_2\sin(\theta_1+\theta_2)\\
L_1\cos\theta_1+L_2\cos(\theta_1+\theta_2)&L_2\cos(\theta_1+\theta_2)
\end{pmatrix}.}
$$

Studentene skal bruke matrisen, ikke utlede den med flervariabel kjerneregel.

In [ ]:
def J_matrise(theta):
    theta1, theta2 = theta
    return np.array([
        [-L1*np.sin(theta1) - L2*np.sin(theta1 + theta2),
         -L2*np.sin(theta1 + theta2)],
        [ L1*np.cos(theta1) + L2*np.cos(theta1 + theta2),
          L2*np.cos(theta1 + theta2)]
    ])


theta_stjerne = np.deg2rad(np.array([35.0, 70.0]))
J_stjerne = J_matrise(theta_stjerne)
_, p_stjerne = framoverkinematikk(theta_stjerne)

print("Arbeidsstilling i radianer:", theta_stjerne)
print("Gripepunkt:", p_stjerne)
print("J*:
", J_stjerne)
print("det(J*) =", np.linalg.det(J_stjerne))

## B.1 Hva betyr kolonnene i $J^*$?

Første kolonne er den omtrentlige gripepunktbevegelsen når bare ledd 1 endres med én radian i den lokale modellen. Andre kolonne har tilsvarende betydning for ledd 2.

Bruk små vinkelendringer, for eksempel $0.5^\circ$, og sammenlign

- den lineære prediksjonen $J^*\Delta\theta$,
- den faktiske forskjellen fra framoverkinematikken.

In [ ]:
delta = np.deg2rad(0.5)

for delta_theta in [np.array([delta, 0.0]), np.array([0.0, delta])]:
    delta_p_lineær = ...
    _, p_ny = framoverkinematikk(theta_stjerne + delta_theta)
    delta_p_faktisk = ...

    print("delta_theta =", delta_theta)
    print("lineær prediksjon =", delta_p_lineær)
    print("faktisk endring =", delta_p_faktisk)
    print("feil =", ...)

## Oppgave B1: Framoverproblem

Beregn $\Delta p$ for

$$
\Delta\theta=
\begin{pmatrix}1^\circ\\-0.5^\circ\end{pmatrix}.
$$

Kontroller resultatet med den ikke-lineære framoverkinematikken.

In [ ]:
delta_theta = np.deg2rad(np.array([1.0, -0.5]))
delta_p = ...

_, p_ny = framoverkinematikk(theta_stjerne + delta_theta)
print("Lineær endring:", delta_p)
print("Faktisk endring:", p_ny - p_stjerne)

## B.2 Lokalt inversproblem

En liten ønsket endring av gripepunktet gir det lineære systemet

$$
\boxed{J^*\Delta\theta=\Delta p.}
$$

Dette er ikke global invers kinematikk. Løsningen gjelder bare for små bevegelser rundt den valgte arbeidsstillingen.

## Oppgave B2: Finn leddendringen

La ønsket bevegelse være

$$
\Delta p=
\begin{pmatrix}5\ \mathrm{mm}\\-3\ \mathrm{mm}\end{pmatrix}.
$$

Løs systemet og kontroller med framoverkinematikken.

In [ ]:
delta_p_ønsket = np.array([5e-3, -3e-3])
delta_theta_beregnet = ...

_, p_kontroll = framoverkinematikk(theta_stjerne + delta_theta_beregnet)

print("Leddendring i grader:", np.rad2deg(delta_theta_beregnet))
print("Ønsket gripepunktendring:", delta_p_ønsket)
print("Faktisk gripepunktendring:", p_kontroll - p_stjerne)

## B.3 Inverterbarhet og singularitet

For toleddet arm gjelder

$$
\boxed{\det J=L_1L_2\sin\theta_2.}
$$

Matrisen er singulær når armen er helt utstrakt eller helt foldet:

$$\theta_2=0\quad\text{eller}\quad\theta_2=\pi.$$

Nær en slik stilling kan en liten ønsket gripepunktbevegelse kreve svært store leddendringer.

## Oppgave B3: Samme oppgave i tre arbeidsstillinger

Sammenlign

1. $\theta_2=70^\circ$,
2. $\theta_2=10^\circ$,
3. $\theta_2=1^\circ$,

med samme $\theta_1$ og samme $\Delta p$.

Registrer

- $|\det J|$,
- $\|\Delta\theta\|$,
- feilen i den lineære prediksjonen.

In [ ]:
theta2_liste = np.deg2rad([70.0, 10.0, 1.0])

for theta2 in theta2_liste:
    theta_test = np.array([theta_stjerne[0], theta2])
    J_test = J_matrise(theta_test)
    delta_theta_test = ...
    _, p0_test = framoverkinematikk(theta_test)
    _, p1_test = framoverkinematikk(theta_test + delta_theta_test)

    print("theta2 =", np.rad2deg(theta2))
    print("|det J| =", abs(np.linalg.det(J_test)))
    print("||delta_theta|| =", np.linalg.norm(delta_theta_test))
    print("faktisk feil =", np.linalg.norm((p1_test-p0_test)-delta_p_ønsket))

## B.4 Hovedretninger for små bevegelser

Vi undersøker den symmetriske matrisen

$$
\boxed{S=JJ^T.}
$$

Egenvektorene til $S$ er to ortogonale retninger i gripepunktplanet. Egenverdiene forteller hvor sterkt leddbevegelser kan gi gripepunktbevegelse i disse retningene.

Hvis én egenverdi er svært liten, er robotarmen nær en singularitet.

## Oppgave B4: Diagonaliser $JJ^T$

Beregn egenverdier og egenvektorer i den valgte arbeidsstillingen. Tegn egenretningene som piler fra gripepunktet.

In [ ]:
S = ...
egenverdier_S, egenvektorer_S = ...

print("S =
", S)
print("Egenverdier:", egenverdier_S)
print("Egenvektorer:
", egenvektorer_S)
print("Kontroll diagonaliserbarhet:
",
      eigenvectors_check if False else "fyll inn kontroll")

fig, ax = plt.subplots()
tegn_robot(theta_stjerne, ax=ax)
for j in range(2):
    retning = egenvektorer_S[:, j]
    lengde = 0.25*np.sqrt(eigenvalues_dummy) if False else 0.20
    ax.arrow(p_stjerne[0], p_stjerne[1],
             lengde*retning[0], lengde*retning[1],
             width=0.005, length_includes_head=True)
plt.show()

## Oppgave B5: Følsomhet nær singularitet

Beregn egenverdiene til $JJ^T$ for en serie verdier av $\theta_2$ som nærmer seg null. Plott den minste egenverdien og $|\det J|$.

Forklar hvorfor begge størrelsene varsler at det lokale inverse problemet blir følsomt.

In [ ]:
theta2_grid = np.deg2rad(np.linspace(1.0, 90.0, 180))
minste_egenverdi = []
determinanter = []

for theta2 in theta2_grid:
    J_test = J_matrise([theta_stjerne[0], theta2])
    S_test = ...
    verdier = ...
    minste_egenverdi.append(...)
    determinanter.append(...)

# Lag plott.

# Del C: Koblet dynamikk i robotleddene

La

$$
q(t)=
\begin{pmatrix}q_1(t)\\q_2(t)\end{pmatrix}
$$

være små vinkelavvik fra arbeidsstillingen:

$$q=\theta-\theta^*.$$

Vi bruker en forenklet servomodell

$$
\boxed{M\ddot q+D\dot q+Kq=\tau(t).}
$$

Her er

- $M$ en treghetsmatrise,
- $D$ en dempningsmatrise,
- $K$ en regulator- eller stivhetsmatrise,
- $\tau(t)$ et påført moment.

Matrisene gis som modellparametre. Studentene skal ikke utlede dem fra full robotmekanikk.

## C.1 Modellmatriser

Vi begynner med

$$
M=
\begin{pmatrix}1.8&0\\0&1.2\end{pmatrix},
$$

$$
D=
\begin{pmatrix}5.0&0.8\\0.8&4.0\end{pmatrix},
$$

$$
K=
\begin{pmatrix}45&12\\12&30\end{pmatrix}.
$$

Kryssleddene betyr at bevegelsen i ett ledd påvirker momentet i det andre.

In [ ]:
M = np.array([
    [1.8, 0.0],
    [0.0, 1.2]
])

D = np.array([
    [5.0, 0.8],
    [0.8, 4.0]
])

K = np.array([
    [45.0, 12.0],
    [12.0, 30.0]
])

print("Egenverdier til M:", np.linalg.eigvalsh(M))
print("Egenverdier til D:", np.linalg.eigvalsh(D))
print("Egenverdier til K:", np.linalg.eigvalsh(K))

## C.2 Førsteordenssystem

Sett

$$v=\dot q.$$

Tilstandsvektoren er

$$X=(q_1,q_2,v_1,v_2)^T.$$

Da blir

$$
\boxed{
\begin{aligned}
\dot q&=v,\\
M\dot v&=\tau(t)-Dv-Kq.
\end{aligned}}
$$

Ved hvert tidspunkt løser vi

$$M\dot v=\tau-Dv-Kq$$

med `np.linalg.solve`.

## Oppgave C1: Implementer modellen og Euler

Fullfør høyresiden og Eulers metode.

In [ ]:
def nullmoment(t):
    return np.zeros(2)


def robot_ode(t, X, momentfunksjon=nullmoment):
    q = X[:2]
    v = X[2:]
    dq = v
    tau = momentfunksjon(t)
    dv = ...
    return np.concatenate([dq, dv])


def euler_system(f, X0, sluttid, h):
    n = int(round(sluttid/h))
    t = np.linspace(0.0, n*h, n + 1)
    X = np.zeros((n + 1, len(X0)))
    X[0] = X0

    for j in range(n):
        X[j + 1] = ...

    return t, X

## C.3 Fri respons etter en posisjonsfeil

Robotarmen begynner med

$$
q(0)=
\begin{pmatrix}5^\circ\\-3^\circ\end{pmatrix},
\qquad
\dot q(0)=0.
$$

Sett $\tau=0$ og simuler responsen.

In [ ]:
X0 = np.concatenate([
    np.deg2rad(np.array([5.0, -3.0])),
    np.zeros(2)
])

t_C, X_C = euler_system(
    lambda t, X: robot_ode(t, X, nullmoment),
    X0,
    sluttid=8.0,
    h=0.002
)

q_C = X_C[:, :2]
v_C = X_C[:, 2:]

fig, ax = plt.subplots(2, 1, sharex=True)
ax[0].plot(t_C, np.rad2deg(q_C[:, 0]), label="q1")
ax[0].plot(t_C, np.rad2deg(q_C[:, 1]), label="q2")
ax[0].set_ylabel("Vinkelavvik i grader")
ax[0].legend()
ax[0].grid()

ax[1].plot(t_C, np.rad2deg(v_C[:, 0]), label="v1")
ax[1].plot(t_C, np.rad2deg(v_C[:, 1]), label="v2")
ax[1].set_xlabel("Tid s")
ax[1].set_ylabel("Vinkelhastighet grader/s")
ax[1].legend()
ax[1].grid()
plt.show()

## Oppgave C2: Gripepunktfeilen

Den lokale modellen gir

$$
\boxed{\Delta p(t)\approx J^*q(t).}
$$

Beregn og plott $\Delta x(t),\Delta y(t)$. Tegn også gripepunktets lokale bane.

In [ ]:
delta_p_C = ...

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(t_C, 1000*delta_p_C[:, 0], label="delta x")
ax[0].plot(t_C, 1000*delta_p_C[:, 1], label="delta y")
ax[0].set_xlabel("Tid s")
ax[0].set_ylabel("Gripepunktfeil i mm")
ax[0].legend()
ax[0].grid()

ax[1].plot(1000*delta_p_C[:, 0], 1000*delta_p_C[:, 1])
ax[1].set_xlabel("delta x i mm")
ax[1].set_ylabel("delta y i mm")
ax[1].axis("equal")
ax[1].grid()
plt.show()

## Oppgave C3: Kontroller den lokale tilnærmingen

Beregn også gripepunktet med den fulle framoverkinematikken:

$$p_{full}(t)=p(\theta^*+q(t)).$$

Sammenlign dette med

$$p_{lokal}(t)=p^*+J^*q(t).$$

Hvordan påvirkes feilen dersom startavvikene dobles?

In [ ]:
p_full = []
for q in q_C:
    _, P = framoverkinematikk(theta_stjerne + q)
    p_full.append(P)
p_full = np.array(p_full)

p_lokal = ...
lokaliseringsfeil = ...

print("Maksimal feil i lokal modell:", np.max(lokaliseringsfeil), "m")
# Lag plott.

## C.4 Trinnrespons mot en ny leddstilling

La ønsket leddavvik være

$$q_d=
\begin{pmatrix}4^\circ\\2^\circ\end{pmatrix}.
$$

En enkel regulert modell er

$$
\boxed{M\ddot q+D\dot q+K(q-q_d)=0.}
$$

Dette tilsvarer momentet

$$\tau=Kq_d.$$

In [ ]:
q_d = np.deg2rad(np.array([4.0, 2.0]))

def trinnmoment(t):
    return K @ q_d

X0_trinn = np.zeros(4)
t_trinn, X_trinn = euler_system(
    lambda t, X: robot_ode(t, X, trinnmoment),
    X0_trinn,
    sluttid=8.0,
    h=0.002
)

q_trinn = X_trinn[:, :2]

# Plott leddrespons og gripepunktrespons.

## Oppgave C4: Demping og oversving

Gjenta trinnresponsen med

- mindre demping,
- den opprinnelige dempingen,
- større demping.

Sammenlign

- oversving,
- innstillingstid,
- samspillet mellom de to leddene,
- gripepunktbanen.

# Del D: Diagonalisering og normale moder

Denne delen forenkler modellen litt for å gjøre variabelbyttet tydelig.

Anta

$$M=mI,\qquad D=cI,$$

og behold den symmetriske stivhetsmatrisen $K$.

Siden $K$ er symmetrisk, finnes en ortogonal matrise $P$ slik at

$$
\boxed{K=P\Lambda P^T.}
$$

Gjør variabelbyttet

$$q=Pz.
$$

Da blir

$$
m\ddot z+c\dot z+\Lambda z=P^T\tau(t).
$$

Uten ytre moment er systemet to uavhengige andreordens ODE-er.

## Oppgave D1: Diagonaliser $K$

Finn egenverdier og ortonormale egenvektorer. Kontroller

$$P^TP=I$$

og

$$K=P\Lambda P^T.$$

In [ ]:
egenverdier_K, P = ...
Lambda = ...

print("Egenverdier:", egenverdier_K)
print("P:
", P)
print("P^T P:
", ...)
print("P Lambda P^T:
", ...)

## Oppgave D2: Tolk egenvektorene

Hver egenvektor beskriver en kombinert leddbevegelse. Tegn robotarmen i

- arbeidsstillingen,
- arbeidsstillingen pluss en liten forskyvning langs første egenvektor,
- arbeidsstillingen pluss en liten forskyvning langs andre egenvektor.

Beskriv om leddene beveger seg samme vei eller motsatt vei i hver mode.

In [ ]:
amplitude = np.deg2rad(5.0)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for j in range(2):
    tegn_robot(theta_stjerne, ax=ax[j], label="arbeidsstilling")
    tegn_robot(theta_stjerne + amplitude*P[:, j], ax=ax[j], label=f"mode {j+1}")
    ax[j].legend()
plt.show()

## D.3 To frakoblede ODE-er

Med $M=mI$, $D=cI$ og $\tau=0$ får vi

$$
m\ddot z_1+c\dot z_1+\lambda_1z_1=0,
$$

$$
m\ddot z_2+c\dot z_2+\lambda_2z_2=0.
$$

Studentene kan løse disse som vanlige andreordens ODE-er og transformere tilbake med

$$q=Pz.$$

## Oppgave D3: Sammenlign modal og direkte simulering

Bruk

$$m=1.5,\qquad c=4.5.$$

1. Transformér starttilstanden med $z(0)=P^Tq(0)$.
2. Løs de to uavhengige ODE-ene med Euler.
3. Transformér tilbake med $q=Pz$.
4. Sammenlign med direkte Euler på det koblede systemet.

In [ ]:
m_modal = 1.5
c_modal = 4.5

q0 = np.deg2rad(np.array([5.0, -3.0]))
z0 = ...

# Implementer de to modal-ODE-ene og sammenlign med direkte system.

# Valgfri del E: Fra ønsket gripepunkt til dynamisk respons

Et ønsket lokalt gripepunktavvik $\Delta p_d$ kan omformes til et ønsket leddavvik ved

$$
\boxed{J^*q_d=\Delta p_d.}
$$

Deretter brukes servomodellen

$$
M\ddot q+D\dot q+K(q-q_d)=0.
$$

Dette gir en enkel kjede:

1. ønsket punkt i $xy$-koordinater,
2. lineært system for ønsket leddstilling,
3. ODE for robotens faktiske respons,
4. framoverkinematikk for virkelig gripepunktbane.

## Oppgave E1: Lokal gripepunktkommando

Velg

$$
\Delta p_d=
\begin{pmatrix}20\ \mathrm{mm}\\10\ \mathrm{mm}\end{pmatrix}.
$$

Beregn $q_d$, simuler responsen og sammenlign

- ønsket sluttpunkt,
- lokalt beregnet sluttpunkt,
- sluttpunktet fra full framoverkinematikk.

In [ ]:
delta_p_d = np.array([20e-3, 10e-3])
q_d_lokal = ...

print("Ønsket leddavvik i grader:", np.rad2deg(q_d_lokal))

# Simuler med moment tau = K @ q_d_lokal og kontroller sluttpunktet.

# Modellkritikk

Diskuter minst fem punkter:

- Robotarmen er plan og har bare to ledd.
- Kalibreringen antas affin.
- Kameraet gir et ferdig identifisert punkt.
- Den lokale kinematikken gjelder bare for små leddendringer.
- Matrisen $J$ gis uten utledning.
- Nær singularitet blir det lokale inverse problemet følsomt.
- Leddgrenser og kollisjoner er utelatt.
- Matrisene $M,D,K$ er konstante.
- Full robotdynamikk har vanligvis vinkelavhengig treghet og flere ikke-lineære ledd.
- Friksjon, slark og metning i motorene er utelatt.
- Framoverkinematikken brukes bare til posisjon, ikke endeverktøyets orientering.
- Euler-metoden kan kreve liten steglengde.

## Mulige videreføringer

- flere kalibreringspunkter og minste kvadraters metode,
- treleddet plan robotarm,
- leddgrenser og hindringer,
- global invers kinematikk,
- vinkelavhengig massematrise,
- måledata fra en undervisningsrobot,
- sammenligning med Watt-koblingens mekanisk fastlagte bane.

# Oppsummering

Skriv en kort rapport der du forklarer

1. hvordan kamerakalibreringen ga to $3\times3$-systemer,
2. hva den homogene transformasjonsmatrisen gjør,
3. hvordan $J^*$ koblet små leddendringer til små gripepunktbevegelser,
4. hva kolonnene i $J^*$ betyr,
5. hvorfor liten determinant gir et følsomt inversproblem,
6. hva egenretningene til $JJ^T$ beskriver,
7. hvordan ledddynamikken ble skrevet som et førsteordens ODE-system,
8. hvordan gripepunktfeilen ble beregnet fra leddfeilen,
9. hvordan diagonalisering frakoblet den forenklede dynamikken,
10. hvilke deler av modellen som må forbedres for en virkelig robot.

## Referanser for videre lesning

Prosjektet bruker standard framoverkinematikk og lokal hastighetsmatrise for en plan toleddet robotarm. Studentene trenger ikke lese eksterne kilder for å gjennomføre prosjektet.